# Download và chuẩn hóa VizWiz-VQA trên Kaggle

Notebook tải ảnh và annotation VizWiz từ nguồn chính thức, sau đó chuyển đổi sang định dạng `generic_vqa` của SelTDA. Cần bật **Internet** trong Kaggle.

Mặc định notebook tải train và validation. Test chính thức không có nhãn công khai nên chỉ tải khi `INCLUDE_TEST = True`; việc đánh giá accuracy trong repo sử dụng validation.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/fantastichaha11/SelTDA.git'
BRANCH = 'feat/pseudo-label-filter'
REPO_DIR = Path('/kaggle/working/SelTDA')
OUTPUT_ROOT = Path('/kaggle/working/vizwiz')

INCLUDE_TEST = False
INCLUDE_UNANSWERABLE = True
FORCE_DOWNLOAD = False
DELETE_DOWNLOAD_ARCHIVES = False
CREATE_ARCHIVE = False
ARCHIVE_PATH = Path('/kaggle/working/vizwiz_seltda.zip')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Output root: {OUTPUT_ROOT}')

In [ ]:
import subprocess
import sys

if not (REPO_DIR / '.git').exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f'Using existing repository: {REPO_DIR}')

required_scripts = [
    REPO_DIR / 'scripts/download_vizwiz.py',
    REPO_DIR / 'convert_vizwiz.py',
]
missing_scripts = [str(path) for path in required_scripts if not path.is_file()]
if missing_scripts:
    raise FileNotFoundError(f'Missing repository scripts: {missing_scripts}')
print(f'Repository ready: {REPO_DIR}')

In [ ]:
download_command = [
    sys.executable,
    'scripts/download_vizwiz.py',
    '--output-root',
    str(OUTPUT_ROOT),
]
if INCLUDE_TEST:
    download_command.append('--include-test')
if FORCE_DOWNLOAD:
    download_command.append('--force')

print('Running:', ' '.join(download_command))
subprocess.run(download_command, cwd=REPO_DIR, check=True)
print('VizWiz download and extraction completed.')

In [ ]:
convert_command = [
    sys.executable,
    'convert_vizwiz.py',
    '--vizwiz-root',
    str(OUTPUT_ROOT),
    '--output-root',
    str(OUTPUT_ROOT),
]
if not INCLUDE_UNANSWERABLE:
    convert_command.append('--exclude-unanswerable')

print('Running:', ' '.join(convert_command))
subprocess.run(convert_command, cwd=REPO_DIR, check=True)
print('VizWiz conversion completed.')

In [ ]:
import json

required_outputs = [
    OUTPUT_ROOT / 'train.json',
    OUTPUT_ROOT / 'val.json',
    OUTPUT_ROOT / 'answer_list.json',
    OUTPUT_ROOT / 'vizwiz_val_metadata.json',
    OUTPUT_ROOT / 'images/train',
    OUTPUT_ROOT / 'images/val',
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError(f'Missing converted VizWiz outputs: {missing_outputs}')

def load_json(path):
    with path.open(encoding='utf-8') as file:
        return json.load(file)

train_records = load_json(OUTPUT_ROOT / 'train.json')
val_records = load_json(OUTPUT_ROOT / 'val.json')
answer_list = load_json(OUTPUT_ROOT / 'answer_list.json')
val_metadata = load_json(OUTPUT_ROOT / 'vizwiz_val_metadata.json')

if not train_records or not val_records or not answer_list:
    raise ValueError('Converted train, val, or answer list is empty.')

all_question_ids = [row['question_id'] for row in train_records + val_records]
if len(all_question_ids) != len(set(all_question_ids)):
    raise ValueError('question_id values are not unique across train and val.')

for split_name, records in [('train', train_records), ('val', val_records)]:
    for row in records:
        required_fields = {'image', 'question', 'question_id', 'answer'}
        if not required_fields <= row.keys():
            raise ValueError(f'Malformed {split_name} record: {row}')
        if not row['answer']:
            raise ValueError(f'Empty answers in {split_name}: {row}')
        image_path = OUTPUT_ROOT / 'images' / row['image']
        if not image_path.is_file():
            raise FileNotFoundError(f'Missing image: {image_path}')

missing_metadata = [
    row['question_id'] for row in val_records if str(row['question_id']) not in val_metadata
]
if missing_metadata:
    raise ValueError(f'Missing validation metadata for question IDs: {missing_metadata[:10]}')

print(f'Train records: {len(train_records):,}')
print(f'Validation records: {len(val_records):,}')
print(f'Candidate answers: {len(answer_list):,}')
print(f'Validation metadata rows: {len(val_metadata):,}')
if INCLUDE_TEST:
    test_images = list((OUTPUT_ROOT / 'images/test').glob('*'))
    print(f'Test images (no public labels): {len(test_images):,}')

In [ ]:
from IPython.display import display
from PIL import Image

sample = val_records[0]
sample_image_path = OUTPUT_ROOT / 'images' / sample['image']
print(json.dumps(sample, indent=2, ensure_ascii=False))
display(Image.open(sample_image_path).convert('RGB'))

In [ ]:
import shutil
from zipfile import ZIP_DEFLATED, ZipFile

if DELETE_DOWNLOAD_ARCHIVES:
    downloads_dir = OUTPUT_ROOT / 'downloads'
    if downloads_dir.is_dir():
        shutil.rmtree(downloads_dir)
        print(f'Removed downloaded ZIP files: {downloads_dir}')

if CREATE_ARCHIVE:
    files_to_archive = [
        OUTPUT_ROOT / 'train.json',
        OUTPUT_ROOT / 'val.json',
        OUTPUT_ROOT / 'answer_list.json',
        OUTPUT_ROOT / 'vizwiz_val_metadata.json',
    ]
    for split_name in ('train', 'val', 'test'):
        split_dir = OUTPUT_ROOT / 'images' / split_name
        if split_dir.is_dir():
            files_to_archive.extend(path for path in split_dir.rglob('*') if path.is_file())
    with ZipFile(ARCHIVE_PATH, 'w', compression=ZIP_DEFLATED, allowZip64=True) as archive:
        for path in files_to_archive:
            archive.write(path, path.relative_to(OUTPUT_ROOT))
    print(f'Archive created: {ARCHIVE_PATH}')
else:
    print(f'Dataset is ready at: {OUTPUT_ROOT}')
    print('Set CREATE_ARCHIVE=True only if a ZIP artifact is needed; VizWiz images are large.')